# F4 — Análisis de resultados SIMCE 2025
**4° Básico · Matemática**

Este cuaderno implementa el análisis final acordado para responder las tres preguntas de investigación:

1. ¿Existen diferencias estadísticamente significativas en los puntajes promedio según ubicación geográfica?
2. ¿Qué regiones, provincias y comunas concentran establecimientos con desempeño inferior a la referencia nacional?
3. ¿Qué establecimientos y territorios pueden identificarse como potenciales focos de priorización?

### Decisiones metodológicas
- La **unidad de análisis inferencial** es el establecimiento.
- Para comparar establecimientos territorialmente se usa el **promedio simple**.
- Para estimar desempeño territorial y construir la referencia de criticidad se usa el **promedio ponderado por estudiantes**.
- El ICT comunal combina **déficit (50%) + concentración (30%) + exposición (20%)**, normalizados mediante percentiles.
- Se selecciona el **P90 del ICT** como criterio de focalización relativa.
- La sensibilidad se comprueba excluyendo comunas bajo P10 del número de establecimientos.

## 1. Configuración, carga y validaciones

In [ ]:
from pathlib import Path
from importlib import metadata
import sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.oneway import anova_oneway, effectsize_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

candidatos = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidatos if (p/'src/poo.py').is_file() and (p/'data/raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook dentro del repositorio SIMCE con src/ y data/.')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from src.analisis_f4 import (validar_entrada, promedio_ponderado, resumen_criticidad,
    calcular_ict, prueba_territorial, REGION, COMUNA)

SEMILLA = 42
CUANTIL_CRITICO = 0.90
np.random.seed(SEMILLA)
OUT = ROOT/'F4/resultados'
OUT.mkdir(parents=True, exist_ok=True)
paquetes = ['numpy','pandas','matplotlib','nbformat','nbclient','ipykernel','ipython']
versiones = {p: metadata.version(p) for p in paquetes}
print('Python:', platform.python_version(), '| Plataforma:', platform.platform())
print('Entorno virtual:', sys.prefix != sys.base_prefix)
display(pd.DataFrame(versiones.items(), columns=['paquete','version']))
pd.set_option('display.max_columns', 14)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')